# 06 — Pattern Discovery

**Purpose:** Extract interpretable trading patterns using:
1. **Association rules** — frequent co-occurrence of discretised feature states with specific labels
2. **Decision tree surrogate** — extract human-readable if-then rules from model predictions
3. **SHAP analysis** — understand which features drive model decisions
4. **Pattern ranking** — score candidates by frequency, precision, and expected profit

**Goal:** Not to find the best model — to find **interpretable patterns** that can be translated into strategy entry conditions.

**Inputs:** `data/features/{symbol}_labeled_*.parquet`, saved models  
**Outputs:** Pattern candidate table in `reports/`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_paths, get_validation_params
from src.models.explainability import (
    compute_shap_values, shap_summary_df, compute_permutation_importance,
    fit_surrogate_tree, extract_tree_rules
)
from src.models.train_ml import build_models, walk_forward_splits

cfg      = load_config()
paths    = get_paths(cfg, "..")
symbol   = get_symbol(cfg)
val_cfg  = get_validation_params(cfg)

## 1. Load data and best model

In [ ]:
TP_ATR, SL_ATR, HORIZON = 2.0, 1.0, 30
key     = f"tp{TP_ATR}_sl{SL_ATR}_h{HORIZON}"
labeled = pd.read_parquet(paths["features"] / f"{symbol}_labeled_{key}.parquet")

exclude_prefixes = ("open", "high", "low", "close", "tick_volume", "real_volume", "spread",
                    "label_", "barrier_side_", "time_to_exit_", "mfe_", "mae_")
feature_cols = [c for c in labeled.columns if not any(c.startswith(p) for p in exclude_prefixes)]

X = labeled[feature_cols].fillna(0)
y = labeled["label_long"]

valid_mask = labeled[feature_cols].notna().all(axis=1)
X, y = X[valid_mask], y[valid_mask]

print(f"Loaded: X={X.shape}, y distribution:")
print(y.value_counts().sort_index())

# Safe: pkl files were written by this user's own training code on this machine.
# Never load .pkl files received from untrusted or external sources.
model_path = paths["models"] / f"{symbol}_random_forest_long.pkl"
if model_path.exists():
    rf_model = joblib.load(model_path)
    print(f"\nLoaded model: {model_path}")
else:
    print("Model not found — retrain in notebook 05 first.")
    rf_model = None

## 2. Association rules

Discretise continuous features into bins and find frequent item sets that co-occur with label=+1.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Select a small set of interpretable features for association rules
assoc_features = [
    "rsi_14", "macd_hist", "adx", "bb_pct_b",
    "vol_ratio_20_120", "session_xetra_open",
    "session_london_open", "session_ny_open",
    "direction",
]
assoc_features = [c for c in assoc_features if c in X.columns]

# Discretise into 3 bins: low / mid / high
X_disc = X[assoc_features].copy()
for col in ["rsi_14", "adx", "bb_pct_b", "vol_ratio_20_120"]:
    if col in X_disc.columns:
        X_disc[col] = pd.cut(X_disc[col], bins=3, labels=["low", "mid", "high"])

# One-hot encode for mlxtend
X_ohe = pd.get_dummies(X_disc.astype(str))
X_ohe["label_long_1"] = (y.values == 1).astype(int)

# Apriori
freq_items = apriori(X_ohe.astype(bool), min_support=0.01, use_colnames=True, max_len=4)
rules = association_rules(freq_items, metric="confidence", min_threshold=0.4)

# Filter rules where consequent is "label_long_1"
target_rules = rules[
    rules["consequents"].apply(lambda x: "label_long_1" in x)
].copy()
target_rules = target_rules.sort_values("lift", ascending=False)

print(f"Association rules targeting long label=1: {len(target_rules)}")
display(target_rules[["antecedents", "support", "confidence", "lift"]].head(20))

## 3. Decision tree surrogate rules

In [ ]:
if rf_model is not None:
    y_pred = rf_model.predict(X.values)
    surrogate = fit_surrogate_tree(X, y_pred, max_depth=4)
    rules_text = extract_tree_rules(surrogate, list(X.columns))
    print("Surrogate decision tree rules:")
    print(rules_text[:3000])  # first 3000 chars
    
    # Save full rules
    with open("../reports/surrogate_tree_rules.txt", "w") as f:
        f.write(rules_text)
    print("\nFull rules saved → reports/surrogate_tree_rules.txt")
else:
    print("Skipped — no model loaded.")

## 4. SHAP analysis

In [ ]:
if rf_model is not None:
    import shap
    
    # Use last 2000 test bars for SHAP
    X_shap = X.tail(2000)
    shap_values, explainer = compute_shap_values(rf_model, X_shap)
    
    # Summary DataFrame
    shap_df = shap_summary_df(shap_values, list(X_shap.columns), class_idx=1)
    print("Top 20 features by mean |SHAP|:")
    display(shap_df.head(20))
    shap_df.to_csv("../reports/shap_summary.csv", index=False)
    
    # SHAP beeswarm plot (class 1 = long signal)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(
        sv, X_shap,
        max_display=20,
        show=False
    )
    plt.tight_layout()
    plt.savefig("../reports/shap_beeswarm.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("Skipped — no model loaded.")

## 5. Pattern ranking

Build a summary table of discovered patterns scored by: support, precision, lift, and frequency.

In [ ]:
pattern_rows = []

# From association rules
for _, row in target_rules.head(10).iterrows():
    pattern_rows.append({
        "source":    "association_rule",
        "pattern":   str(row["antecedents"]),
        "support":   round(row["support"], 4),
        "confidence": round(row["confidence"], 4),
        "lift":      round(row["lift"], 3),
        "side":      "long",
    })

# From SHAP top features (placeholder — enrich in later notebooks)
if rf_model is not None:
    top_shap_features = shap_df.head(5)["feature"].tolist()
    for feat in top_shap_features:
        pattern_rows.append({
            "source":    "shap_top_feature",
            "pattern":   feat,
            "support":   None,
            "confidence": None,
            "lift":      None,
            "side":      "long",
        })

pattern_df = pd.DataFrame(pattern_rows)
print("\nDiscovered patterns:")
display(pattern_df)

pattern_df.to_csv("../reports/pattern_candidates.csv", index=False)
print("\nSaved → reports/pattern_candidates.csv")